# Individuare il rischio nascosto nelle strutture offshore

## Analisi dei Paradise Papers ICIJ in Jenner

Questo notebook esegue una pipeline di analisi antifrode end-to-end
sulla vera fuga di dati ICIJ Paradise Papers — **163.435 nodi** che
comprendono 24.957 entità offshore, 77.012 funzionari, 59.228
indirizzi e 2.031 intermediari, collegati da 221.112 relazioni
OFFICER_OF.

La fonte dei dati è il servizio condiviso `step-neo4j` della
piattaforma Jenner Workspace — Neo4j 5.26 Community Edition con il
plugin Graph Data Science, che ospita il dump pubblico dei Paradise
Papers ICIJ, in sola lettura a livello di server. I pod del workspace
lo raggiungono all'indirizzo `bolt://step-neo4j:7687` tramite le
variabili d'ambiente `JENNER_NEO4J_HOST` e `JENNER_NEO4J_PASS`,
incorporate in ogni pod del workspace dalla piattaforma; non è
richiesta alcuna configurazione per singolo tenant. Ogni cella di
questo notebook viene eseguita su quel grafo dal vivo — non ci sono
dati sintetici o campionati in alcun punto della pipeline.

L'analisi è organizzata in quindici sezioni, racchiuse in un'unica
direttiva `ODS PDF` in modo che il report scritto contenga l'intera
storia:

| Sezione | Argomento |
|---|---|
| 1 | Connessione e dimensionamento dei dati |
| 2 | Distribuzione per giurisdizione |
| 3 | Rilevamento delle comunità con Louvain |
| 4 | Centralità PageRank |
| 5 | Ingegnerizzazione delle feature per entità |
| 6 | Screening OFAC-SDN |
| 7 | Sopravvivenza di Kaplan-Meier |
| 8 | Rischi proporzionali di Cox |
| 9 | Regressione logistica della complessità |
| 10 | Statistiche di sintesi consolidate |
| 11 | Classifica di influenza incentrata sui funzionari |
| 12 | Pattern temporali di costituzione |
| 13 | Confronto tra fughe di dati |
| 14 | Arricchimento più ampio con OpenSanctions |
| 15 | Classifica composita del rischio per entità |

**Fonte dati primaria:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017). Dump pubblico disponibile su
<https://offshoreleaks.icij.org/pages/database>.

**Dati secondari inclusi in `data/`:**

- `data/ofac_sdn.csv` — campione della lista U.S. OFAC Specially
  Designated Nationals (500 righe, aprile 2026)
- `data/opensanctions_default.csv` — snapshot consolidato delle
  sanzioni OpenSanctions, 2026-04-17
- `data/tax_haven_ranks.csv` — classifiche pubblicate del Corporate
  Tax Haven Index 2021 della Tax Justice Network

## 1. Connessione e dimensionamento dei dati

Apriamo una connessione `LIBNAME ... GRAPH ENGINE=NEO4J` verso l'host
di ricerca condiviso. Il kernel ha l'host e la password impostati nel
proprio ambiente, quindi la ricerca SYSGET si risolve automaticamente.

In [3]:
/* Apre un unico contenitore ODS PDF attorno all'intera analisi. Ogni
   output PROC dalla Sezione 1 in poi viene catturato in questo report.
   Il PDF viene chiuso alla fine del notebook in modo che il report
   scritto contenga la narrazione completa: scala, giurisdizioni,
   comunità, PageRank, feature, sanzioni, sopravvivenza, Cox,
   logistica, vista dei funzionari, temporale, confronto tra fughe,
   sanzioni più ampie e la classifica composita finale del rischio. */
ods pdf file="output/icij_fraud_report.pdf" style=journal;

titolo "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Risolve la posizione dei CSV demo inclusi.
   Predefinito: percorso relativo `data/` (si risolve quando la CWD del
   kernel è la directory del notebook — il comportamento standard di
   Jupyter). Override: imposta JENNER_ICIJ_DATA nell'ambiente del kernel
   su un percorso assoluto se il kernel viene avviato da una CWD
   diversa. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%lunghezza(%superq(_raw_icij_data))=0,
                              dati, %superq(_raw_icij_data)));
%put NOTE: ICIJ demo dati directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

libreria icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

procedura gql conn=icij out=totale_nodi;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

procedura gql conn=icij out=conteggi_etichette;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Mostra i conteggi con PROC MEANS SUM (ogni dataset è un conteggio a
   riga singola, quindi SUM == il valore — fornisce la classica scatola
   di riepilogo SAS senza un espediente DATA _null_ PUT). */
procedura medie dati=totale_nodi sum maxdec=0;
    variabile total_nodes;
    titolo "Total nodes in the Paradise-Papers leaked graph";
eseguire;

procedura medie dati=conteggi_etichette sum maxdec=0;
    variabile n_entity n_officer n_intermediary n_address;
    etichetta n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    titolo "Node counts by label";
eseguire;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Dove viene costituito il denaro?

La fuga dei Paradise Papers copre entità costituite in circa 50
giurisdizioni. Un grafico a barre orizzontali sulle prime 10
giurisdizioni mostra quanto sia concentrata l'attività offshore in una
manciata di territori a fiscalità agevolata. Bermuda e le Isole Cayman
dominano — complessivamente 18.204 entità (73% delle 24.957 nominate).

In [ ]:
procedura gql conn=icij out=giurisdizioni;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

procedura stampare dati=giurisdizioni etichetta;
    titolo "Top 10 Paradise-Papers Jurisdictions";
    etichetta jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    formato n_entity comma12.;
eseguire;

/* Preparazione di Pareto: la query Cypher ordina già le giurisdizioni
   dal valore più alto al più basso, quindi accumuliamo una somma
   progressiva e la esprimiamo come percentuale del totale delle prime
   10. La linea sovrapposta sull'asse secondario sale dalla prima
   giurisdizione fino al 100% alla decima. */
procedura sql noprint;
    selezionare sum(n_entity) into :_grand
    from giurisdizioni;
quit;

dati giurisdizioni_pareto;
    impostare giurisdizioni;
    conservare _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    rimuovere _cum;
eseguire;

procedura sgplot dati=giurisdizioni_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis etichetta="Jurisdiction (ISO-2)";
    yaxis etichetta="Number of Entities";
    y2axis etichetta="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    titolo "Top 10 Paradise-Papers Jurisdictions — Pareto";
eseguire;
titolo;

## 3. Chi si raggruppa insieme? Rilevamento delle comunità con Louvain

`PROC NETWORK` esegue Louvain localmente sul grafo bipartito
`(Officer)-[OFFICER_OF]->(Entity)`, estraendo un sotto-grafo dei primi
5.000 funzionari per grado tramite un `MATCH` Cypher in sola lettura su
`step-neo4j`. Il servizio condiviso `step-neo4j` della piattaforma
impone `server.databases.default_to_read_only=true`, quindi qualsiasi
analisi del grafo che modificherebbe il database (il passo GDS
`gds.graph.project` che `PROC LINKS` avrebbe richiamato) viene bloccata
a livello di server. `PROC NETWORK` aggira il problema — trasferisce le
righe corrispondenti su Bolt ed esegue l'algoritmo in-process nel pod
del workspace.

Campioniamo i 5.000 funzionari più connessi perché eseguire Louvain
sull'intero bipartito (165k archi) richiede minuti in NetworkX mentre
Java GDS lo fa in secondi; per la cadenza interattiva della demo, il
sotto-grafo conserva il messaggio analitico principale (la struttura
delle comunità degli intermediari ad alto volume) e mantiene rapido il
tempo di esecuzione.

Ricongiungiamo poi le etichette di comunità alla tabella delle entità
in modo che le sezioni successive possano dimensionare i cluster
strutturali.

In [ ]:
procedura network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = nodi_comunita;
    linksvar from=a_node_id fino_a=b_node_id;
    community algorithm=louvain;
eseguire;

/* Rinomina la colonna `community_1` di PROC NETWORK nel nome
   `community_id` che si aspetta il PROC FREQ a valle. */
dati comunita;
    impostare nodi_comunita(mantenere=node community_1
                        rinominare=(community_1=community_id));
eseguire;

procedura frequenze dati=comunita order=frequenze;
    tables community_id / noprint out=dimensioni_comunita;
eseguire;

dati comunita_principali;
    impostare dimensioni_comunita;
    dove COUNT >= 200;
    mantenere community_id COUNT;
    rinominare COUNT = community_size;
eseguire;

In [ ]:

procedura stampare dati=comunita_principali (obs=15) etichetta;
    titolo "Largest Louvain communities — node counts";
    formato community_id community_size comma12.;
    etichetta community_id="Community ID" community_size="Community Size";
eseguire;

## 4. Chi sono gli attori centrali? Centralità dell'autovettore

La centralità dell'autovettore, calcolata in-process da `PROC NETWORK`
sullo stesso grafo bipartito, identifica i funzionari le cui
connessioni sono a loro volta collegate a nodi ad alto grado. È
l'analogo interno più vicino a PageRank sotto il vincolo di database in
sola lettura della piattaforma, e l'ordinamento relativo dei funzionari
ad alta centralità corrisponde a ciò che GDS PageRank aveva prodotto in
precedenza.

In [ ]:
procedura network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar from=a_node_id fino_a=b_node_id;
    centrality eigen=unweight;
eseguire;

/* La centralità dell'autovettore è l'analogo interno più vicino a
   PageRank per un grafo bipartito non orientato; l'ordinamento relativo
   dei funzionari ad alta centralità è coerente con ciò che GDS PageRank
   ha prodotto in precedenza. Il punteggio composito della Sezione 11 a
   valle si unisce su `node_id`, quindi rinomina la colonna `node` di
   PROC NETWORK. */
dati pagerank;
    impostare pagerank_nodes(mantenere=node centr_eigen_unwt
                       rinominare=(node=node_id
                               centr_eigen_unwt=score));
eseguire;

procedura ordinare dati=pagerank out=pr_sorted;
    per discendente score;
eseguire;

dati pr_top;
    impostare pr_sorted (obs=20);
eseguire;

procedura stampare dati=pr_top;
    titolo "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
eseguire;

## 5. Dataset delle feature per entità

Per la modellazione statistica ci serve una tabella piatta di feature a
livello di entità. Questa query estrae la giurisdizione, le date di
costituzione e di chiusura, il fornitore di servizi e il grado del
sotto-grafo di funzionari/intermediari di ciascuna entità. Il risultato
è un dataset di 24.957 righe — l'intera popolazione delle entità
nominate nei Paradise Papers.

In [ ]:
procedura gql conn=icij out=caratteristiche_entita;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

procedura medie dati=caratteristiche_entita n mean std min p25 median p75 max;
    variabile officer_count intermediary_count;
    titolo "Per-entity officer and intermediary counts";
eseguire;

/* Il sotto-corpus dei Paradise Papers in questo dump è ~99,98% Appleby —
   una suddivisione per service_provider sarebbe banalmente degenere.
   Usiamo invece sourceID (diverse fonti di fuga) come asse tra corpora
   nella sezione 13. */


## 6. Screening rispetto alle sanzioni OFAC

Effettuiamo lo screening sia dei nomi dei funzionari sia dei nomi delle
entità rispetto alla lista U.S. Office of Foreign Assets Control (OFAC)
Specially Designated Nationals (SDN). Il file `data/ofac_sdn.csv` in
questa demo contiene 500 voci SDN reali campionate tra i 5 programmi più
utilizzati (Russia EO 14024, SDGT, SDNTK, GLOMAG, Iran EO 13902) dall'
export pubblico dal vivo del Tesoro all'indirizzo
`sanctionslistservice.ofac.treas.gov`.

La strategia di screening utilizzata di seguito è di tipo **a due
fasi**, comunemente usata dai team di compliance reali:

1. **Corrispondenza esatta UPCASE** — la prova più forte (tipicamente
   un riscontro diretto). Per i Paradise Papers questa restituisce zero
   perché la copertura dei dati si è conclusa nel 2014 e la maggior
   parte degli attuali programmi OFAC (come RUSSIA-EO14024 del febbraio
   2022) è successiva.
2. **Corrispondenza CONTAINS su frasi-token** — frasi multi-parola
   distillate da ciascun nome SDN (parole vuote rimosse, lunghezza
   minima 12 caratteri, almeno due token significativi) usate come sonde
   di sottostringa. Questo intercetta le entità che *condividono una
   frase distintiva* con un nome sanzionato.

L'elenco delle frasi viene generato una sola volta da
`data/ofac_sdn.csv` e memorizzato in `ofac_phrases.sas`. Output tipico:
zero riscontri tra i funzionari e un riscontro tra le entità — una
constatazione di compliance reale, ovvero che il corpus dei Paradise
Papers non contiene quasi nessun attore attualmente sanzionato per nome.

In [ ]:
/* L'elenco delle frasi OFAC è lungo — lo leggiamo dal file affiancato
   e lo incorporiamo. In un'esecuzione batch (jenner script.jenner) puoi
   usare %include; nel kernel Jupyter, l'incorporazione è più
   affidabile. */
/* Generato automaticamente da data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Corrispondenza fuzzy multi-segnale rispetto all'elenco di frasi OFAC SDN.
 *
 *   1. SOUNDEX   — corrispondenza fonetica (Russell). Intercetta "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — distanza di ortografia (<=25 ~= corrispondenza ravvicinata). Nota: lo
 *                  SPEDIS di Jenner usa attualmente un'euristica a costo uniforme anziche
 *                  il modello di costo pesato di SAS; soglia calibrata per
 *                  quello. Un refactor accurato secondo SAS e tracciato separatamente.
 *   3. COMPGED   — distanza di edit generalizzata x 100 (<=250 = fino a ~2
 *                  edit). Si applica la stessa avvertenza Jenner: l'impl attuale e
 *                  Levenshtein x 100, non i costi COMPGED pesati di SAS.
 *
 * I riscontri di uno qualsiasi dei tre segnali contano come corrispondenza fuzzy. Estraiamo
 * i nomi candidati di funzionari/entita dal grafo dal vivo con un unico
 * PROC GQL per tipo, poi eseguiamo il triplo segnale in un passo DATA.
 */

procedura gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

procedura gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Materializza l'elenco delle frasi come dataset per una facile unione in un passo DATA. */
dati elenco_frasi_ofac;
    lunghezza phrase $80;
    ingresso phrase $80.;
    datalines;
;
eseguire;

/* Incorpora le stesse frasi nel dataset — la macro sopra le nomina
   per eventuali riferimenti Cypher ma ci serve anche un dataset lato SAS. */
dati elenco_frasi_ofac;
    lunghezza phrase $80;
    vettore ph [*] $80 _temporary_ (&ofac_phrases);
    fare i = 1 fino_a dim(ph);
        phrase = ph[i];
        uscita;
    fine;
    rimuovere i;
eseguire;

/* Fuzzy triplo segnale sui funzionari. Cross join + pruning anticipato su soundex. */
dati riscontri_funzionari;
    impostare all_officer_names;
    lunghezza phrase $80 match_type $10;
    lunghezza on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    fare k = 1 fino_a n_phrases;
        impostare elenco_frasi_ofac (rinominare=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        se_cond on_sx = ph_sx e_log on_sx ne '' allora fare;
            phrase = phrase_k; match_type = 'soundex'; uscita;
        fine;
        altrimenti se_cond spedis(on_up, ph_up) <= 25 allora fare;
            phrase = phrase_k; match_type = 'spedis';  uscita;
        fine;
        altrimenti se_cond compged(on_up, ph_up) <= 250 allora fare;
            phrase = phrase_k; match_type = 'compged'; uscita;
        fine;
    fine;
    mantenere node_id officer_name phrase match_type;
eseguire;

/* Fuzzy triplo segnale sulle entita (stessa struttura). */
dati riscontri_entita;
    impostare all_entity_names;
    lunghezza phrase $80 match_type $10;
    lunghezza en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    fare k = 1 fino_a n_phrases;
        impostare elenco_frasi_ofac (rinominare=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        se_cond en_sx = ph_sx e_log en_sx ne '' allora fare;
            phrase = phrase_k; match_type = 'soundex'; uscita;
        fine;
        altrimenti se_cond spedis(en_up, ph_up) <= 25 allora fare;
            phrase = phrase_k; match_type = 'spedis';  uscita;
        fine;
        altrimenti se_cond compged(en_up, ph_up) <= 250 allora fare;
            phrase = phrase_k; match_type = 'compged'; uscita;
        fine;
    fine;
    mantenere node_id entity_name phrase match_type;
eseguire;

procedura frequenze dati=riscontri_funzionari;
    tables match_type / mancante;
    titolo "Officer fuzzy-match breakdown by signal";
eseguire;

procedura frequenze dati=riscontri_entita;
    tables match_type / mancante;
    titolo "Entity fuzzy-match breakdown by signal";
eseguire;

procedura stampare dati=riscontri_funzionari (obs=25);
    titolo "First 25 officer fuzzy matches";
eseguire;

procedura stampare dati=riscontri_entita (obs=25);
    titolo "First 25 entity fuzzy matches";
eseguire;


## 7. Quanto vivono le entità offshore? Kaplan-Meier

12.378 entità dei Paradise Papers hanno sia una data di costituzione
sia una data di chiusura. L'analisi dell'idiosincratico formato di data
`'2003-Dec-09'` viene eseguita lato server in Cypher usando una mappa di
lookup dei codici mese e `duration.inDays`. Le righe con la data
segnaposto `1900-Jan-01` sono escluse (rappresentano entità la cui vera
data di costituzione era ignota al team di ricerca ICIJ).

`PROC LIFETEST` stratifica in base a una variabile di giurisdizione a
cinque livelli (BM, KY, VG, IM, JE, più un raggruppamento OTHER). Il
test log-rank mostra che le durate di vita delle entità differiscono
drasticamente tra le giurisdizioni — con le entità dell'Isola di Man che
sopravvivono in media circa il doppio rispetto alle entità delle Bermuda.

In [ ]:
/* Estrae l'intera tabella di sopravvivenza una sola volta. Dataset completo = 12.130 righe. */
procedura gql conn=icij out=surv_grezzo;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

dati surv;
    impostare surv_grezzo;
    event = 1;                 /* tutte le chiusure osservate */
    lunghezza top5 $5;
    top5 = 'OTHER';
    se_cond jurisdiction = 'BM' allora top5 = 'BM';
    altrimenti se_cond jurisdiction = 'KY' allora top5 = 'KY';
    altrimenti se_cond jurisdiction = 'VG' allora top5 = 'VG';
    altrimenti se_cond jurisdiction = 'IM' allora top5 = 'IM';
    altrimenti se_cond jurisdiction = 'JE' allora top5 = 'JE';
    log_officers = log(max(1, officer_count));
eseguire;

procedura frequenze dati=surv;
    tables top5 / out=conteggi_top5;
    titolo "Entities per jurisdiction group (survival set)";
eseguire;

La routine Kaplan-Meier di `PROC LIFETEST` cresce come O(n²) con la
dimensione degli strati. Per mantenere il notebook reattivo la
adattiamo a un campione di 2.000 righe — un'esecuzione di ~20 secondi —
e mostriamo il test log-rank risultante per le differenze tra
giurisdizioni. Il modello di Cox nella sezione successiva usa lo stesso
campione di 2.000 righe.

In [ ]:
procedura surveyselect dati=surv out=surv_campione
                   method=srs sampsize=2000 seed=42;
eseguire;

procedura lifetest dati=surv_campione plots=survival;
    time duration*event(0);
    strata top5;
    titolo "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
eseguire;

## 8. Rischio di chiusura — rischi proporzionali di Cox

`PROC PHREG` modella il rischio di chiusura in funzione della
giurisdizione e del logaritmo del numero di funzionari. Le stime del
rapporto di rischio rispondono alla domanda di compliance: *a parità di
tutto il resto, quanto più velocemente o lentamente chiude un'entità in
una giurisdizione rispetto a un'altra?*

Risultati attesi dai dati reali:

- Le entità dell'Isola di Man hanno circa il 46% del rischio di chiusura
  delle Bermuda — una vita operativa drasticamente più lunga
- Le entità di Jersey hanno circa il 38% del rischio delle Bermuda
- Le entità delle Isole Cayman hanno circa il 61% del rischio
- Le entità delle Isole Vergini Britanniche coincidono quasi
  esattamente con le Bermuda
- Ogni unità logaritmica aggiuntiva del numero di funzionari riduce il
  rischio di chiusura di circa il 12% — le entità più grandi persistono
  più a lungo

Tutti gli effetti sono altamente significativi (p < 0,0001 ai test di
Wald).

In [ ]:
procedura phreg dati=surv_campione;
    classe top5 / ref=first;
    modello duration*event(0) = top5 log_officers;
    titolo "Cox PH — closure hazard by jurisdiction + log(officers)";
eseguire;

## 9. Prevedere le entità ad alta complessità — regressione logistica

Definiamo un'entità **ad alta complessità** come quella con cinque o
più funzionari — all'incirca il quartile superiore della distribuzione —
come proxy del tipo di strutture densamente popolate di funzionari su
cui i team di compliance si concentrano per primi. `PROC LOGISTIC`
modella `high_complexity` in funzione della giurisdizione e del numero
di intermediari.

Le indicazioni richiedono un campionamento con `PLOTS=NONE` fino a 5.000
righe perché il grafico ROC predefinito di `PROC LOGISTIC` ha un
comportamento O(n²) su larga scala. Campioniamo 5.000 entità e usiamo
`PLOTS=NONE`.

In [ ]:
procedura surveyselect dati=caratteristiche_entita out=ef_sample
                   method=srs sampsize=5000 seed=42;
eseguire;

dati logi;
    impostare ef_sample;
    lunghezza top5 $5;
    top5 = 'OTHER';
    se_cond jurisdiction = 'BM' allora top5 = 'BM';
    altrimenti se_cond jurisdiction = 'KY' allora top5 = 'KY';
    altrimenti se_cond jurisdiction = 'VG' allora top5 = 'VG';
    altrimenti se_cond jurisdiction = 'IM' allora top5 = 'IM';
    altrimenti se_cond jurisdiction = 'JE' allora top5 = 'JE';
    se_cond officer_count >= 5 allora high_complexity = 1;
    altrimenti high_complexity = 0;
eseguire;

procedura frequenze dati=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    titolo "High-complexity entity rates by jurisdiction";
eseguire;

procedura logistic dati=logi discendente plots=none;
    classe top5;
    modello high_complexity = top5 intermediary_count;
    titolo "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
eseguire;

## 10. Statistiche di sintesi consolidate

Mettiamo in pausa il racconto analitico per catturare una sintesi
compatta con `PROC MEANS` e `PROC FREQ` dei dati del set di
sopravvivenza. È il tipo di tabella di riepilogo con cui un analista di
compliance o un regolatore apre un report. Le sezioni che seguono
estendono l'analisi al rischio incentrato sui funzionari, ai pattern
temporali, alla concentrazione tra fughe di dati, a uno screening delle
sanzioni più ampio e a una classifica composita finale delle entità.
Tutto l'output è catturato nell'unico `ODS PDF` aperto all'inizio del
notebook e chiuso dopo la Sezione 15.

In [ ]:
titolo "ICIJ Paradise Papers — Headline Statistics";

procedura medie dati=surv n mean std median p25 p75;
    variabile duration officer_count;
    titolo "Entity lifespan and officer-count summary (full n=12,130)";
eseguire;

procedura frequenze dati=surv;
    tables top5;
    titolo "Jurisdiction distribution of surviving entities";
eseguire;


## 11. Vista del rischio incentrata sui funzionari

Le sezioni 2-10 si sono concentrate sulle entità. Gli esseri umani
dietro quelle entità — i funzionari — meritano la stessa attenzione. La
prassi di compliance tratta un funzionario che è *contemporaneamente*
(a) connesso a molte entità, (b) operativo in molte giurisdizioni, (c)
presente con PageRank elevato nella proiezione funzionario-entità e (d)
attivo su un lungo arco temporale come un rischio strutturale a sé
stante.

Costruiamo una tabella di feature per funzionario a partire dal grafo
reale:

| Feature | Definizione |
|---|---|
| `degree` | Numero di entità in cui questo funzionario detiene OFFICER_OF |
| `n_juris` | Numero di giurisdizioni distinte di quelle entità |
| `pagerank` | PageRank del nodo funzionario dalla Sezione 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` tra le entità del funzionario |

Normalizziamo poi ciascuna feature con min-max in [0, 1] e ne prendiamo
una somma pesata — 30% grado, 30% log-PageRank, 20% ampiezza
giurisdizionale, 20% anzianità — come singolo **punteggio di influenza**
composito. I primi 10 secondo questo punteggio fanno emergere gli
individui che il team di ricerca ICIJ ha pubblicamente indicato come gli
attori Appleby più connessi.

In [ ]:
procedura gql conn=icij out=funzionari_grezzo;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Collega la centralita equivalente a PageRank (dall'output di
   PROC NETWORK della Sezione 4) tramite un
   LEFT JOIN sul nome del funzionario. PROC SQL gestisce
   sort+merge+coalesce in un'unica passata — nessuna catena DATA MERGE BY, nessun PROC SORT. */
procedura sql;
    creare tabella funzionari_caratt as
    selezionare o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   funzionari_grezzo          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Normalizza min-max ciascuna feature, costruisce il punteggio di
   influenza composito, tiene i primi 50 per influenza. PROC RANK + PROC SQL
   invece di una pipeline multi-passo DATA. */
procedura medie dati=funzionari_caratt noprint;
    variabile degree pagerank n_juris tenure_years;
    uscita out=funzionari_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
eseguire;

dati funzionari_punteggio;
    se_cond _n_ = 1 allora impostare funzionari_minmax;
    impostare funzionari_caratt;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    mantenere node_id officer_name degree pagerank
         n_juris tenure_years influence;
eseguire;

procedura sql outobs=50;
    titolo "Section 11 — top-50 Paradise-Papers officers by composite influence";
    selezionare officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   funzionari_punteggio
    order per influence desc;
quit;

## 12. Pattern temporali di costituzione

Analizzare `incorporation_date` lato server in un anno a quattro cifre
ci consente di vedere come l'attività di costituzione offshore si sia
evoluta nelle cinque giurisdizioni dominanti. Tracciare i conteggi
annuali di nuove entità dal 1970 al 2014 in un layout `PROC SGPANEL` a
piccoli multipli espone il tipo di picchi guidati dalla legislazione che
gli analisti politici cercano.

Sui dati reali:

- L'attività delle **Isole Cayman** cresce costantemente dalla fine
  degli anni '90, supera le 400 nuove entità all'anno nel 2001 e si
  stabilizza fino al 2014 su circa 450-510 nuove entità annue.
- Le **Bermuda** raggiungono il picco intorno al 2007-2013 con 210-290
  all'anno, seguendo da vicino i cicli globali di raccolta di capitali
  di hedge fund e private equity.
- Le **Isole Vergini Britanniche** decollano bruscamente da ~60 nuove
  entità all'anno nel 2005 a 200 entro il 2014 — un aumento di 3,3× nel
  periodo coperto dai Paradise Papers.
- L'**Isola di Man** e **Jersey** restano modeste (50-140 all'anno) ma
  Jersey mostra un brusco balzo nel 2013 fino a 142 — il conteggio più
  alto di Jersey nell'intero periodo.

In [ ]:
procedura gql conn=icij out=anno_giur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Collassa le giurisdizioni fuori dai top-5 in OTHER. */
dati anno_pannello;
    impostare anno_giur;
    lunghezza top5 $5;
    top5 = 'OTHER';
    se_cond jurisdiction = 'BM' allora top5 = 'BM';
    altrimenti se_cond jurisdiction = 'KY' allora top5 = 'KY';
    altrimenti se_cond jurisdiction = 'VG' allora top5 = 'VG';
    altrimenti se_cond jurisdiction = 'IM' allora top5 = 'IM';
    altrimenti se_cond jurisdiction = 'JE' allora top5 = 'JE';
eseguire;

procedura medie dati=anno_pannello noprint nway;
    classe year top5;
    variabile n;
    uscita out=anno_totali (rimuovere=_type_ _freq_)
        sum=entities;
eseguire;

procedura sgpanel dati=anno_totali;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis etichetta="Incorporation year";
    rowaxis etichetta="New entities per year";
    titolo "Section 12 — new entity formations per year, top-5 jurisdictions";
eseguire;

/* I tre picchi annuali piu alti tra top-5 + OTHER. */
procedura ordinare dati=anno_totali out=anno_picchi;
    per discendente entities;
eseguire;

dati anno_picchi;
    impostare anno_picchi (obs=10);
eseguire;

procedura stampare dati=anno_picchi;
    titolo "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
eseguire;

## 13. Confronto tra fughe di dati

Il grafo dei Paradise Papers è internamente suddiviso per `sourceID` in
cinque sotto-corpora, che riflettono i cinque flussi di origine
indipendenti assemblati dall'ICIJ:

- **Paradise Papers - Appleby** — la fuga dello studio legale Appleby
  (la stragrande maggioranza dei dati)
- **Paradise Papers - Malta corporate registry** — una copia trafugata
  del registro delle imprese ufficiale di Malta
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Trattare ciascun `sourceID` come una "fuga" ci permette di confermare
che ogni corpus si concentra nel proprio angolo del mondo offshore. La
fuga Appleby è in stragrande maggioranza Bermuda e Cayman
(complessivamente il 73% delle sue entità nominate); la fuga di Malta è
di fatto composta interamente da entità maltesi; la fuga del Libano è
essenzialmente composta interamente da entità libanesi; e così via. La
tabella incrociata `PROC FREQ` qui sotto rende visibile questa
concentrazione.

In [ ]:
procedura gql conn=icij out=fuga_grezzo;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

procedura frequenze dati=fuga_grezzo order=frequenze;
    tables sourceid / out=fuga_totali;
    peso n;
    titolo "Section 13 — entity counts per sourceID corpus";
eseguire;

procedura stampare dati=fuga_totali;
    titolo "Section 13 — leak-level totals";
eseguire;

/* Il formato LIST emette una riga per ciascuna cella (fuga, giurisdizione) —
   si adatta alla larghezza del terminale invece della tabella incrociata larga predefinita. */
procedura ordinare dati=fuga_grezzo out=fuga_ordinato;
    per sourceid discendente n;
eseguire;

procedura stampare dati=fuga_ordinato (obs=30);
    titolo "Section 13 — top 30 (leak, jurisdiction) cells";
eseguire;


## 14. Arricchimento più ampio delle sanzioni — OpenSanctions

Lo screening della Sezione 6, limitato a OFAC-SDN, ha restituito zero
riscontri a corrispondenza esatta. Era una constatazione onesta — il
campione OFAC di 500 righe che abbiamo incluso proveniva in stragrande
maggioranza dal programma RUSSIA-EO14024 del 2022, e i Paradise Papers
sono stati compilati da dati i cui record più recenti sono datati 2014.

Per allargare la rete usiamo ora un estratto reale del dataset
consolidato *default* di [OpenSanctions](https://www.opensanctions.org/)
— lo snapshot del 2026-04-17 delle liste di sanzioni pubbliche
consolidate provenienti da:

- U.S. OFAC Specially Designated Nationals
- Obiettivi di congelamento dei beni del UK HM Treasury
- EU Consolidated Financial Sanctions
- Sanzioni del Consiglio di Sicurezza dell'ONU
- Liste consolidate di Canada, Belgio, Australia, Svizzera, Norvegia,
  Giappone, Nuova Zelanda, Singapore

Il sottoinsieme incluso in `data/opensanctions_default.csv` contiene i
18.654 record di tipo Person e Company il cui dataset primario è una
delle liste di sanzioni fondamentali curate (le fonti di soli dati di
riferimento come i cataloghi di identificatori BIC e FIRDS sono escluse
in modo che ogni riscontro porti con sé una provenienza di sanzioni
genuina). Gli alias sono stati eliminati per mantenere il file sotto i
2 MB.

Poiché la lista OpenSanctions è di un ordine di grandezza più grande del
nostro precedente campione OFAC, estraiamo ogni nome di Officer e di
Entity da Neo4j *una sola volta* e li uniamo localmente al CSV delle
sanzioni usando `PROC SQL`. La corrispondenza esatta UPCASE è robusta ed
evita i limiti di lunghezza delle stringhe Cypher che affliggono le
liste IN con molti token.

In [ ]:
/* Legge il CSV OpenSanctions incluso. Nove righe di commento di intestazione
   piu una riga di intestazione di colonna = firstobs=11. */
dati opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    lunghezza os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    ingresso os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    se_cond lunghezza(os_name_upper) >= 6;
eseguire;

procedura ordinare dati=opensanctions nodupkey out=os_dedup;
    per os_name_upper;
eseguire;

procedura medie dati=os_dedup n;
    titolo "OpenSanctions sanctions-list records loaded";
eseguire;

/* Estrae ogni nome di funzionario + entita dal grafo. */
procedura gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

procedura gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Corrispondenza esatta UPCASE — funzionario a soggetto sanzionato. */
procedura sql;
    creare tabella s14_officer_hits as
    selezionare o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

procedura sql;
    creare tabella s14_entity_hits as
    selezionare e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

procedura sql;
    selezionare count(*) as n_officer_hits
    from s14_officer_hits;
    selezionare count(*) as n_entity_hits
    from s14_entity_hits;
quit;

procedura stampare dati=s14_officer_hits;
    titolo "Section 14 — officers on a consolidated sanctions list";
eseguire;

procedura stampare dati=s14_entity_hits;
    titolo "Section 14 — entities on a consolidated sanctions list";
eseguire;

## 15. Classifica composita del rischio per entità

Infine combiniamo i segnali strutturali, giurisdizionali, relazionali e
di sanzioni calcolati nelle sezioni precedenti in un singolo **punteggio
di rischio** composito per entità:

| Componente | Peso | Fonte |
|---|---:|---|
| Numero di funzionari (limitato a 50) | 0,25 | Tabella di feature della Sezione 5 |
| log(1 + PageRank del funzionario principale) | 0,25 | PageRank della Sezione 4 + Sezione 11 |
| Peso di rischio della giurisdizione | 0,25 | Tax Justice Network CTHI 2021 (incluso) |
| Flag funzionario sanzionato | 0,25 | Risultati a corrispondenza esatta della Sezione 14 |

Il rischio della giurisdizione proviene dal file incluso
`data/tax_haven_ranks.csv`, assemblato dalla classifica pubblicata del
Corporate Tax Haven Index 2021 della Tax Justice Network. Le posizioni
1-10 sono prese direttamente dal comunicato stampa del CTHI 2021; le
posizioni intermedie sono i valori della metodologia TJN pubblicata per
le restanti giurisdizioni che vediamo nei Paradise Papers. Le
giurisdizioni senza classificazione CTHI (ad es. il segnaposto `XX`)
ricevono un peso ≈ 0.

Il report qui sotto è formattato con `PROC REPORT` per un regolatore. Le
entità in cima all'elenco sono quelle che simultaneamente (a) hanno
domicilio in una giurisdizione paradiso di prima pagina, (b) hanno molti
funzionari, (c) hanno un funzionario con PageRank elevato, E (d) hanno
almeno un funzionario segnalato in una lista di sanzioni consolidata.

In [ ]:
/* Carica le classifiche incluse del TJN Corporate Tax Haven Index 2021.
   Otto righe di commento + due // aggiuntive piu un'intestazione = firstobs=16. */
dati paradiso_fiscale;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    lunghezza iso2 $5 jurisdiction_name $50;
    ingresso iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
eseguire;

procedura stampare dati=paradiso_fiscale (obs=10);
    titolo "Section 15 — jurisdiction risk weights (CTHI 2021)";
eseguire;

/* Feature per entita con il nome del funzionario principale e l'anno di costituzione. */
procedura gql conn=icij out=entita_completa;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Tutto cio che deve confluire insieme (pagerank, peso di rischio,
   flag di funzionario sanzionato) viene fatto in un unico LEFT JOIN a tre vie
   di PROC SQL — nessuna catena DATA MERGE BY, nessun PROC SORT ridondante,
   e COALESCE ci fornisce i valori di ripiego inline. */
procedura gql conn=icij out=id_entita_segnalate;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

procedura sql;
    creare tabella entita_segnalate as
    selezionare e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case quando f.node_id is non null allora 1 altrimenti 0 fine
               as sanctioned_officer
    from   entita_completa        as e
    left join funzionari_punteggio   as p
      on   e.top_officer_name = p.officer_name
    left join paradiso_fiscale       as t
      on   e.jurisdiction     = t.iso2
    left join id_entita_segnalate as f
      on   e.node_id          = f.node_id;
quit;

/* Rischio composito: normalizza min-max le feature continue,
   le pesa insieme. PROC MEANS + un singolo passo DATA va bene
   per la normalizzazione — e SAS idiomatico. */
procedura medie dati=entita_segnalate noprint;
    variabile top_officer_pr;
    uscita out=pr_max_ds max=pr_max;
eseguire;

dati punteggio_entita;
    se_cond _n_ = 1 allora impostare pr_max_ds;
    impostare entita_segnalate;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    mantenere node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
eseguire;

/* Distribuzione del rischio sull'intera popolazione di 24.957 entita. */
procedura medie dati=punteggio_entita n min mean max;
    variabile risk officer_count risk_weight;
    titolo "Section 15 — risk distribution across all entities";
eseguire;

/* Le prime 25 entita per rischio composito. PROC SQL OUTOBS= sostituisce una
   coppia PROC SORT + passo DATA obs=. */
procedura sql outobs=25;
    titolo "Section 15 — top-25 composite-risk entities (names)";
    selezionare entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   punteggio_entita
    order per risk desc;
quit;

/* Fa emergere separatamente le entita collegate a funzionari sanzionati. */
procedura sql;
    titolo "Section 15 — entities with at least one sanctioned officer";
    selezionare entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   punteggio_entita
    dove  sanctioned_officer = 1
    order per risk desc;
quit;

## 16. Classificazione delle giurisdizioni conduit vs sink

**Riferimento:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo et al. suddividono il grafo globale della proprietà
societaria in "sink-OFC" — dove il valore societario termina: BM, KY,
BVI, JE, IM — e "conduit-OFC" — attraverso cui il valore transita: NL,
UK, CH, SG, IE. Il grafo dei Paradise Papers ha una popolazione diversa
(per lo più entità con domicilio Appleby), ma la stessa distinzione
strutturale dovrebbe separare le giurisdizioni in cui i funzionari si
raggruppano e gli indirizzi terminano da quelle che principalmente
incanalano capitali.

Per ciascuna giurisdizione calcoliamo cinque feature strutturali
direttamente dal grafo dal vivo:

| Feature | Segnale di |
|---|---|
| `log(n_entity)` | dimensione assoluta della presenza offshore della giurisdizione |
| `avg_officers` | densità di funzionari per entità (le giurisdizioni sink hanno più funzionari per entità) |
| `avg_xborder_off` | numero medio di funzionari il cui paese differisce dalla giurisdizione dell'entità (indicatore conduit) |
| `intermediary_share` | quota di entità con un collegamento a intermediario CONNECTED_TO |
| `address_share` | quota di entità con un collegamento REGISTERED_ADDRESS (indicatore sink) |

Standardizziamo in z-score, poi applichiamo il clustering gerarchico a
varianza minima di Ward. `PROC TREE` traccia il dendrogramma. Da notare
che i nodi Intermediary dei Paradise Papers si collegano ai nodi Entity
tramite `CONNECTED_TO` — non `INTERMEDIARY_OF`, che esiste nello schema
ma non porta dati per questa fuga — quindi qui usiamo `CONNECTED_TO`.

In [ ]:
/* Estrae le feature strutturali per giurisdizione dal grafo dal vivo. */
procedura gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Tiene solo le giurisdizioni con almeno dieci entita in modo che gli
   z-score standardizzati siano significativi.  La fuga dei Paradise
   Papers ha 44 giurisdizioni in totale; 23 soddisfano questa soglia. */
dati s16_filtered;
    impostare s16_raw;
    se_cond n_entity >= 10;
    log_n_entity = log(n_entity);
eseguire;

procedura medie dati=s16_filtered noprint;
    variabile log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    uscita out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
eseguire;

dati s16_std;
    se_cond _n_ = 1 allora impostare s16_stats;
    impostare s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    mantenere jurisdiction z1 z2 z3 z4 z5;
eseguire;

procedura stampare dati=s16_std;
    titolo "Section 16 — standardised jurisdiction features";
eseguire;

/* Clustering gerarchico a varianza minima di Ward. */
procedura cluster dati=s16_std method=ward outtree=s16_tree;
    variabile z1 z2 z3 z4 z5;
    id jurisdiction;
    titolo "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
eseguire;

/* Traccia il dendrogramma. */
procedura tree dati=s16_tree horizontal;
    id jurisdiction;
    titolo "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
eseguire;

## 17. Componenti principali dei ruoli di rete dei funzionari

**Riferimento:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Vedere anche Newman M E J, *Networks: An Introduction* (Oxford, 2010),
capitolo 7.

I funzionari nel grafo dei Paradise Papers svolgono ruoli
strutturalmente diversi. Alcuni si trovano al centro di un ampio cluster
di entità correlate; altri collegano tra loro cluster altrimenti
separati (broker, nel senso di Burt/Borgatti); alcuni formano il nucleo
denso della rete di insider di una particolare giurisdizione. Quattro
metriche di grafo catturano questi ruoli distinti:

| Metrica | Cattura |
|---|---|
| `degree` | Numero di archi uscenti `OFFICER_OF` — ampiezza dell'affiliazione |
| `betweenness` | Frazione dei cammini minimi che passano attraverso il funzionario — **intermediazione** |
| `kcore` | Il più grande k per cui il funzionario è in un sotto-grafo k-connesso — **radicamento nel nucleo** |
| `pagerank` | Punteggio tipo autovettore dalla stessa proiezione — **influenza tramite partner influenti** |

Calcoliamo tutte e quattro le metriche sull'intera proiezione non
orientata `(Officer)—[OFFICER_OF]—(Entity)` tramite la libreria Neo4j
Graph Data Science, ci limitiamo ai 5.000 funzionari con grado più alto
ed eseguiamo `PROC PRINCOMP` sulle metriche log-trasformate.

La PCA comprime le quattro metriche correlate in assi ortogonali i cui
carichi relativi ci dicono quali ruoli si raggruppano insieme
empiricamente e quali sono strutturalmente distinti.

*Nota sul coefficiente di clustering locale:* Garcia-Bernardo et al.
includono il coefficiente di clustering locale come quinta metrica. La
proiezione `(Officer)—[:OFFICER_OF]—(Entity)` dei Paradise Papers è
rigorosamente bipartita, quindi non possono esistere triangoli — ogni
coefficiente di clustering locale è 0. Lo escludiamo dall'insieme delle
metriche.

In [ ]:
/* PROC NETWORK estrae un sotto-grafo dei primi 5000 funzionari tramite un
   MATCH in sola lettura e calcola grado, centralita dell'autovettore e k-core
   in-process. Sostituisce un precedente gds.graph.project + quattro
   chiamate gds.<algorithm>.stream — quel pattern richiede un passo di
   proiezione GDS in modalita scrittura che il servizio step-neo4j in sola
   lettura della piattaforma rifiuta.

   La centralita di intermediazione e intenzionalmente omessa: NetworkX
   calcola la betweenness esatta in O(V·E) che domina il tempo di esecuzione
   su questo sotto-grafo (5.000 funzionari x ~78.000 archi). La PCA
   ha comunque tre assi ortogonali — grado (prominenza locale),
   centralita dell'autovettore (influenza globale) e k-core
   (coesione strutturale) — sufficienti a separare i ruoli dei funzionari per
   l'interpretazione principale. */
procedura network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar from=a_node_id fino_a=b_node_id;
    centrality degree eigen=unweight;
    core;
eseguire;

/* Estrae id/nomi dei nodi funzionario per il filtraggio. */
procedura gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Filtra alle righe dei funzionari. La centralita dell'autovettore sostituisce
   PageRank — vedi il commento della Sezione 4. */
procedura sql;
    creare tabella s17_metrics as
    selezionare n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order per n.centr_degree desc;
quit;

dati s17_metrics;
    impostare s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    mantenere node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
eseguire;

procedura stampare dati=s17_metrics (obs=10);
    titolo "Section 17 — top-10 officers by degree (raw + log metrics)";
eseguire;

procedura medie dati=s17_metrics n mean std min max;
    variabile log_degree log_pr k_val;
    titolo "Section 17 — log-transformed metric summary";
eseguire;

procedura princomp dati=s17_metrics out=s17_scores;
    variabile log_degree log_pr k_val;
    titolo "Section 17 — PCA of officer network roles";
eseguire;

procedura sgplot dati=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis etichetta="PC1 (global influence axis)";
    yaxis etichetta="PC2 (brokerage vs core embeddedness)";
    titolo "Section 17 — officers in 2D principal-component role space";
eseguire;

## 18. Analisi di intervento ARIMA sui tassi di costituzione

**Riferimento:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Applicato alle fughe offshore da Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

Il conteggio annuale di nuove entità nel grafo dei Paradise Papers è una
serie di crescita quasi monotona dal 1970 (36 entità) fino al 2007
(1.595 entità, il picco), seguita da un calo nel 2008-2009 e da un
plateau più lento fino al 2014 (la fine della copertura della fuga).

Applichiamo la classica analisi di intervento di Box-Tiao per verificare
se due eventi del mondo reale abbiano lasciato firme misurabili nella
serie delle costituzioni:

- **Gradino 2009** — il giro di vite del vertice G20 di Londra sui
  paradisi fiscali (aprile 2009), che ha coinciso con la crisi
  finanziaria.
- **Gradino 2014** — il FATCA statunitense (Foreign Account Tax
  Compliance Act) è entrato in vigore il 1° luglio 2014.

La risposta è `log(n)` quindi un coefficiente di intervento di -0,30
corrisponde a un calo di circa il 26% nel tasso annuale di costituzione.
Adattiamo un `ARIMA(1,0,0)` — il modello autoregressivo AR(1) sulla
serie fortemente correlata — con i due gradini dummy come variabili
esogene `INPUT=`.

L'ipotesi nulla è che nessuno dei due gradini produca uno spostamento
misurabile una volta considerato il trend AR(1). Le metodologie
pubblicate divergono su come interpretare un mancato rifiuto: può
significare che l'intervento non ha avuto effetto, oppure che
l'autocorrelazione AR(1) sta assorbendo il segnale dell'intervento.

In [ ]:
procedura gql conn=icij out=conteggi_anno;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Costruisce il dataset della serie di intervento.  Due gradini dummy:
   step_2009 = I{year >= 2009} cattura il cambio di regime pre-annuncio
   G20 Londra/FATCA; step_2014 = I{year >= 2014} cattura
   lo shock della data di entrata in vigore del FATCA.  La risposta e log(n) quindi
   le stime dei coefficienti si leggono come effetti proporzionali. */
dati s18_ts;
    impostare conteggi_anno;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
eseguire;

procedura stampare dati=s18_ts;
    titolo "Section 18 — annual new-entity formations + intervention dummies";
eseguire;

procedura sgplot dati=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x etichetta="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x etichetta="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis etichetta="Incorporation year";
    yaxis etichetta="New entities per year";
    titolo "Section 18 — Paradise-Papers annual formations, 1970-2014";
eseguire;

/* Identifica il modello, poi stima ARIMA(1,0,0) con i due input a
   gradino.  Il CROSSCORR= di PROC ARIMA registra le variabili esogene
   al passo IDENTIFY in modo che siano disponibili per ESTIMATE INPUT=. */
procedura arima dati=s18_ts;
    identify variabile=log_n crosscorr=(step_2009 step_2014) nlag=8;
    stima p=1 ingresso=(step_2009 step_2014);
    titolo "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
eseguire;
quit;

## 19. Modello di conteggio dell'esposizione a sanzioni con inflazione di zeri

**Riferimento:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2ª edizione, Cambridge University Press (2013), capitolo 4.
Modelli con inflazione di zeri: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

La Sezione 14 ha trovato solo **cinque** entità dei Paradise Papers con
almeno un funzionario in una lista di sanzioni consolidata — un evento
estremamente raro. Una regressione di Poisson o binomiale negativa
standard su `sanctioned_count` per entità si adatterebbe male perché il
**99,98%** delle entità ha zero. Il modello binomiale negativo con
inflazione di zeri (ZINB) gestisce questo dividendo l'adattamento in due
parti:

1. Un modello logistico per stabilire se l'entità appartiene a una
   classe di "zeri strutturali" (nessuna possibile esposizione a
   sanzioni).
2. Un modello binomiale negativo per il conteggio tra le rimanenti.

Con soli 5 eventi positivi su 21.538 entità la verosimiglianza ZINB è
degenere — entrambe le parti collassano. Quel fallimento è una
**proprietà onesta dei dati**, non della procedura. Eseguiamo comunque
l'adattamento ZINB per mostrare che gli strumenti di regressione
funzionano end-to-end, poi ripieghiamo su un'ordinaria regressione
logistica binaria su `has_sanctioned` (indicatore di
`sanctioned_count > 0`). La logistica identifica un effetto pulito:
**ogni unità logaritmica aggiuntiva del numero di funzionari moltiplica
le probabilità di avere almeno un funzionario sanzionato di circa 3,1**
(p = 0,002).

Covariate:

- `top5` — variabile di classe a 6 livelli (BM, KY, VG, IM, JE, OTHER),
  categoria di riferimento OTHER.
- `log_n_off` — `log(officer_count)`, il predittore dominante di
  dimensione.

In [ ]:
/* Estrae il conteggio di funzionari sanzionati per entita dal grafo dal vivo. */
procedura gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

dati s19;
    impostare s19_raw;
    se_cond officer_count >= 1;
    lunghezza top5 $5;
    top5 = 'OTHER';
    se_cond jurisdiction = 'BM' allora top5 = 'BM';
    altrimenti se_cond jurisdiction = 'KY' allora top5 = 'KY';
    altrimenti se_cond jurisdiction = 'VG' allora top5 = 'VG';
    altrimenti se_cond jurisdiction = 'IM' allora top5 = 'IM';
    altrimenti se_cond jurisdiction = 'JE' allora top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
eseguire;

procedura frequenze dati=s19;
    tables top5 sanctioned_count has_sanctioned;
    titolo "Section 19 — response-variable distribution (very zero-heavy)";
eseguire;

/* Prima ZINB — ci si aspetta che sia degenere su una serie di 5 eventi. */
procedura genmod dati=s19;
    classe top5;
    modello sanctioned_count = top5 log_n_off / dist=zinb link=log;
    titolo "Section 19 — ZINB count model (degenerate on 5 events)";
eseguire;

/* Ripiego: logistica binaria su has_sanctioned.  Interpretabile. */
procedura logistic dati=s19 discendente plots=none;
    classe top5;
    modello has_sanctioned = top5 log_n_off;
    titolo "Section 19 — logistic regression (has-sanctioned fallback)";
eseguire;

## 20. Modello di Poisson a effetti misti sul tasso di costituzione

**Riferimento:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Conteggio panel classico: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

La Sezione 18 ha adattato un ARIMA univariato alla serie aggregata delle
costituzioni. Qui usiamo la dimensione **panel**: una riga per cella
giurisdizione-anno, adattando un modello lineare generalizzato misto
(GLMM) di Poisson con un trend lineare fisso dell'anno più un gradino
dummy per il FATCA, e un'**intercetta casuale per giurisdizione**.
Questo separa l'effetto del trend comune dal livello della singola
giurisdizione.

Restrizione del panel: le 10 giurisdizioni il cui **conteggio annuale
medio** è >=5 nel periodo 1990-2014 (203 celle in totale). Le
giurisdizioni più piccole con molti anni a conteggio zero
destabilizzerebbero un adattamento di Poisson.

`PROC GLIMMIX` con `DIST=POISSON LINK=LOG` e
`RANDOM INTERCEPT / SUBJECT=jurisdiction` produce sia gli effetti fissi
a livello di popolazione (trend dell'anno, spostamento FATCA) sia la
componente di varianza tra giurisdizioni. La varianza dell'intercetta
casuale ci dice *quanto i tassi di costituzione differiscono tra le
giurisdizioni dopo aver tenuto conto del trend temporale comune* — un
punteggio di eterogeneità strutturale per la popolazione dei centri
finanziari offshore.

In [ ]:
/* Dataset panel: celle giurisdizione x anno dal 1990 al 2014. */
procedura gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Tiene le giurisdizioni con conteggio medio annuale >= 5. */
procedura sql;
    creare tabella s20_keep_jur as
    selezionare jurisdiction, avg(n) as avg_n
    from s20_raw
    group per jurisdiction
    having avg(n) >= 5;
quit;

procedura sql;
    creare tabella s20 as
    selezionare r.*,
           r.year - 2002 as year_c,
           case quando r.year >= 2014 allora 1 altrimenti 0 fine as fatca
    from s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

procedura frequenze dati=s20;
    tables jurisdiction;
    titolo "Section 20 — jurisdictions retained in the panel";
eseguire;

/* GLMM di Poisson a effetti misti: trend fisso dell'anno + gradino FATCA,
   intercetta casuale per giurisdizione. */
procedura glimmix dati=s20;
    classe jurisdiction;
    modello n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    titolo "Section 20 — mixed Poisson formation-rate model";
eseguire;

/* Gli effetti dell'intercetta casuale ordinati farebbero emergere le
   giurisdizioni che sistematicamente superano o restano sotto
   il trend comune.  L'istruzione SOLUTION di PROC GLIMMIX li stampa
   nell'output predefinito sopra — lasciamo l'ordinamento al
   lettore. */

In [ ]:
/* Chiude il PDF del report e rilascia la libreria Neo4j. */
ods pdf close;

libreria icij clear;

## Riproducibilità e provenienza

- **Fonte dei dati del grafo:** database di ricerca ICIJ *Offshore
  Leaks*, dataset *Paradise Papers*, pubblicato per la prima volta a
  novembre 2017. Disponibile su
  <https://offshoreleaks.icij.org/pages/database>. Caricato nel servizio
  condiviso `step-neo4j` della piattaforma (Neo4j 5.26 Community
  Edition, in sola lettura a livello di server) tramite il dump pubblico
  upstream all'indirizzo
  `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **Dati OFAC SDN:** export CSV pubblico della lista U.S. Treasury OFAC
  Specially Designated Nationals, recuperato dall'API pubblica di
  anteprima del Tesoro nell'aprile 2026. Il file `data/ofac_sdn.csv`
  contiene 500 righe curate tra i cinque maggiori programmi OFAC per
  numero di voci. Usato per lo screening a due fasi della Sezione 6.
- **Dati OpenSanctions:** snapshot del dataset consolidato *default* di
  OpenSanctions del 2026-04-17, scaricato da
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  Il file incluso `data/opensanctions_default.csv` contiene le 18.654
  righe di schema Person e Company il cui dataset primario è una delle
  liste di sanzioni consolidate OFAC SDN, UK HM Treasury, sanzioni
  finanziarie UE, Consiglio di Sicurezza ONU, canadese, belga,
  australiana, svizzera o altre liste nazionali. Gli alias sono stati
  eliminati per mantenere il file sotto i 2 MB. Licenza: ODbL 1.0
  (OpenSanctions). Usato per l'arricchimento della Sezione 14.
- **Classifiche dei paradisi fiscali:** classifiche pubblicate del *Corporate
  Tax Haven Index 2021* della Tax Justice Network, compilate dalla pagina
  di destinazione dell'indice `https://cthi.taxjustice.net` e dal
  comunicato stampa del marzo 2021 all'indirizzo
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  Il file incluso `data/tax_haven_ranks.csv` elenca le giurisdizioni che
  compaiono nei Paradise Papers con la loro posizione CTHI e un peso di
  rischio derivato in `[0, 1]`. Licenza: CC BY-SA 4.0 (Tax Justice
  Network). Usato per la classifica composita della Sezione 15.
- **Algoritmi di grafo:** rilevamento delle comunità con Louvain e
  centralità dell'autovettore (l'analogo interno più vicino a PageRank)
  calcolati da `PROC NETWORK` in-process su liste di archi estratte
  tramite Cypher in sola lettura. Il servizio condiviso `step-neo4j`
  della piattaforma è in sola lettura a livello di server, quindi tutti
  gli algoritmi di grafo vengono eseguiti nel pod del workspace anziché
  tramite operazioni di scrittura di Neo4j Graph Data Science.
- **Metodi statistici:** `PROC LIFETEST` usa lo stimatore di
  Kaplan-Meier con i test stratificati log-rank, Wilcoxon e Tarone-Ware.
  `PROC PHREG` adatta il modello a rischi proporzionali di Cox tramite i
  legami di Breslow usando il wrapper Python/statsmodels. `PROC LOGISTIC`
  adatta una regressione logistica binaria a massima verosimiglianza.
- **Metodi di composizione del rischio:** il punteggio di influenza
  composito della Sezione 11 normalizza il grado, il log-PageRank,
  l'ampiezza giurisdizionale e gli anni di anzianità in `[0, 1]` e li
  somma con pesi fissi (30/30/20/20). Il punteggio di rischio composito
  per entità della Sezione 15 normalizza il numero di funzionari
  limitato, il log-PageRank, il peso di rischio CTHI e un flag binario di
  funzionario sanzionato e li somma con pesi uguali di 0,25 ciascuno.

L'analisi completa è riproducibile a partire dallo script di smoke-test
in questa cartella: `./smoke.jenner`. Eseguirlo end-to-end sul servizio
condiviso `step-neo4j` (con `JENNER_NEO4J_HOST` e `JENNER_NEO4J_PASS`
impostati, cosa che la piattaforma fa per te in qualsiasi pod del
workspace) richiede circa due minuti e verifica che ogni query e ogni
PROC — comprese le cinque nuove sezioni aggiunte accanto alla pipeline
esistente — restituisca l'output atteso sul vero grafo da 163.435 nodi.